## 1. Import Libraries

In [1]:
# Import all required libraries for data preprocessing and exploration.
# pandas       : Data manipulation and tabular analysis
# numpy        : Numerical operations and statistical computation
# matplotlib   : Data visualisation (used in Milestone 3 / Assignment 1)
# seaborn      : Statistical visualisation (used in Milestone 3 / Assignment 1)
# sklearn      : Label encoding and MinMax scaling utilities
# warnings     : Suppresses non-critical library warnings for cleaner output

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load Dataset

In [2]:
# Load the Accidental Drug-Related Deaths dataset.
# Dataset  : Accidental_Drug_Related_Deaths.csv
# Source   : Connecticut Open Data Portal (public domain)
# Each row : One accidental drug-related death case (medical examiner record)
#
# COLAB USERS: Uncomment the two lines below and update the path.
# from google.colab import drive
# drive.mount('/content/drive')

file_location = 'Accidental_Drug_Related_Deaths.csv'
# If using Colab with Drive, replace with your full path, e.g.:
# file_location = '/content/drive/MyDrive/.../Accidental_Drug_Related_Deaths.csv'

data = pd.read_csv(file_location)
df = data.copy()  # Preserve original dataset; all changes made on df

print("Dataset loaded successfully!")
print(f"Total records : {df.shape[0]}")
print(f"Total columns : {df.shape[1]}")

Dataset loaded successfully!
Total records : 11981
Total columns : 48


## 3. Explore Dataset Structure

In [3]:
# Explore the dataset structure before any preprocessing.
# head()     : First 5 rows — shows data format and early records
# tail()     : Last 5 rows — checks data consistency at end of file
# info()     : Column names, data types, and non-null counts
# describe() : Statistical summary of numerical columns
# shape      : Total row and column count (rows, columns)

print("=" * 80)
print("HEAD (first 5 rows):")
print(df.head())

print("\n" + "=" * 80)
print("TAIL (last 5 rows):")
print(df.tail())

print("\n" + "=" * 80)
print("DATASET INFO:")
df.info()

print("\n" + "=" * 80)
print("STATISTICAL SUMMARY:")
print(df.describe())

print("\n" + "=" * 80)
print(f"SHAPE: {df.shape[0]} rows x {df.shape[1]} columns")

HEAD (first 5 rows):
         Date      Date Type   Age     Sex   Race Ethnicity Residence City  \
0  05/29/2012  Date of death  37.0    Male  Black       NaN       STAMFORD   
1  06/27/2012  Date of death  37.0    Male  White       NaN        NORWICH   
2  03/24/2014  Date of death  28.0    Male  White       NaN         HEBRON   
3  12/31/2014  Date of death  26.0  Female  White       NaN         BALTIC   
4  01/16/2016  Date of death  41.0    Male  White       NaN        SHELTON   

  Residence County Residence State Injury City  ... Xylazine Gabapentin  \
0        FAIRFIELD             NaN    STAMFORD  ...      NaN        NaN   
1       NEW LONDON             NaN     NORWICH  ...      NaN        NaN   
2              NaN             NaN      HEBRON  ...      NaN        NaN   
3              NaN             NaN         NaN  ...      NaN        NaN   
4        FAIRFIELD              CT     SHELTON  ...      NaN        NaN   

  Opiate NOS Heroin/Morph/Codeine Other Opioid Any Opioid O

## 4. Identify Missing Values

In [4]:
# Count and display missing values per column.
# isna().sum()   : Absolute count of missing (NaN) values per column
# Percentage     : Proportion of missing values relative to total rows
# Output guides which imputation strategy to apply per column type.

missing_count = df.isna().sum()
missing_pct = (df.isna().sum() / len(df) * 100).round(2)

print("Columns with Missing Values (absolute count):")
print(missing_count[missing_count > 0])

print("\n" + "=" * 80)
print("Missing Value Percentage by Column (descending):")
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

Columns with Missing Values (absolute count):
Age                                  2
Sex                                  9
Race                                57
Ethnicity                         9416
Residence City                     596
Residence County                  1260
Residence State                   1988
Injury City                        178
Injury County                     3334
Injury State                      3029
Injury Place                       358
Description of Injury              807
Death City                        2784
Death County                      3891
Death State                       5108
Location                          1349
Location if Other                10787
Manner of Death                      9
Other Significant Conditions     10782
Heroin                            8403
Heroin death certificate (DC)    11241
Cocaine                           7403
Fentanyl                          3932
Fentanyl Analogue                11007
Oxycodone         

## 5. Handle Missing Values – Age (Median Imputation)

In [5]:
# Age is a continuous numerical column with only 2 missing values (0.02%).
# Median imputation is chosen because it is robust to skewed distributions
# and outliers, unlike mean imputation. (Week 1 Practical 2 technique)
# Preserving all records avoids sampling bias in downstream analyses.

missing_before = df['Age'].isna().sum()
print(f"Missing Age values before : {missing_before}")

df['Age'] = df['Age'].fillna(df['Age'].median())

missing_after = df['Age'].isna().sum()
print(f"Missing Age values after  : {missing_after}")
print(f"Median value applied      : {df['Age'].median()}")
print(f"\u2713 Filled {missing_before - missing_after} missing Age value(s)")

Missing Age values before : 2
Missing Age values after  : 0
Median value applied      : 44.0
✓ Filled 2 missing Age value(s)


## 6. Handle Missing Values – Sex and Race (Mode Imputation)

In [6]:
# Sex (9 missing, 0.08%) and Race (57 missing, 0.5%) are categorical columns.
# Mode imputation replaces missing entries with the most frequent category,
# minimising distortion to the existing category distribution.
# mode()[0] selects the first (most common) mode value.
# (Week 1 Practical 2 technique)

for col in ['Sex', 'Race']:
    missing_before = df[col].isna().sum()
    mode_value = df[col].mode()[0]
    df[col] = df[col].fillna(mode_value)
    missing_after = df[col].isna().sum()
    print(f"{col}:")
    print(f"  Missing before : {missing_before}")
    print(f"  Mode applied   : '{mode_value}'")
    print(f"  Missing after  : {missing_after}")
    print()

Sex:
  Missing before : 9
  Mode applied   : 'Male'
  Missing after  : 0

Race:
  Missing before : 57
  Mode applied   : 'White'
  Missing after  : 0



## 7. Handle Missing Values – Location Columns (Absolute Fill: 'Unknown')

In [7]:
# Nine location columns have 11-42% missing values.
# Deletion would remove thousands of records, introducing sampling bias.
# Absolute fill with 'Unknown' (Week 1 Practical 2 technique) preserves
# all 11,981 records while explicitly marking incomplete geographic data.
# 'Unknown' also serves as a valid category in clustering/classification.

location_columns = [
    'Residence City', 'Residence County', 'Residence State',
    'Injury City',    'Injury County',    'Injury State',
    'Death City',     'Death County',     'Death State'
]

print("Filling location columns with 'Unknown':")
for col in location_columns:
    if col in df.columns:
        before = df[col].isna().sum()
        df[col] = df[col].fillna('Unknown')
        print(f"  {col:<25} : {before} \u2192 0 missing")

print("\n\u2713 All location columns filled")

Filling location columns with 'Unknown':
  Residence City            : 596 → 0 missing
  Residence County          : 1260 → 0 missing
  Residence State           : 1988 → 0 missing
  Injury City               : 178 → 0 missing
  Injury County             : 3334 → 0 missing
  Injury State              : 3029 → 0 missing
  Death City                : 2784 → 0 missing
  Death County              : 3891 → 0 missing
  Death State               : 5108 → 0 missing

✓ All location columns filled


## 8. Handle Missing Values – Drug Columns (Binary Transformation: blank→0, Y→1)

In [8]:
# Drug columns contain 'Y' (detected) or blank/NaN (not detected/not tested).
# IMPORTANT: A blank value in a toxicology report does NOT mean missing data.
# It means the substance was absent or untested in that case.
# These blank values carry meaningful information and must NOT be deleted.
#
# Transformation: blank/NaN -> '0' (not detected), 'Y' -> '1' (detected)
# This creates binary indicators suitable for all four mining algorithms.

drug_columns = [
    'Heroin', 'Cocaine', 'Fentanyl', 'Fentanyl Analogue', 'Oxycodone',
    'Oxymorphone', 'Ethanol', 'Hydrocodone', 'Benzodiazepine', 'Methadone',
    'Meth/Amphetamine', 'Amphet', 'Tramad', 'Hydromorphone',
    'Morphine (Not Heroin)', 'Xylazine', 'Gabapentin', 'Opiate NOS', 'Any Opioid'
]

print("Applying binary transformation to drug columns (blank \u2192 0, Y \u2192 1):")
for col in drug_columns:
    if col in df.columns:
        df[col] = df[col].fillna('0')
        df[col] = df[col].replace('Y', '1')
        print(f"  \u2713 {col}")

print("\nAll drug columns processed")

Applying binary transformation to drug columns (blank → 0, Y → 1):
  ✓ Heroin
  ✓ Cocaine
  ✓ Fentanyl
  ✓ Fentanyl Analogue
  ✓ Oxycodone
  ✓ Oxymorphone
  ✓ Ethanol
  ✓ Hydrocodone
  ✓ Benzodiazepine
  ✓ Methadone
  ✓ Meth/Amphetamine
  ✓ Amphet
  ✓ Tramad
  ✓ Hydromorphone
  ✓ Morphine (Not Heroin)
  ✓ Xylazine
  ✓ Gabapentin
  ✓ Opiate NOS
  ✓ Any Opioid

All drug columns processed


## 9. Find and Remove Duplicate Records

In [9]:
# Duplicate records inflate frequencies and bias model training.
# duplicated().sum() checks how many exact duplicate rows exist.
# drop_duplicates() removes them if found.
# (Week 1 Practical 2 technique)

duplicate_count = df.duplicated().sum()
print(f"Duplicate records found: {duplicate_count}")

if duplicate_count > 0:
    print("\nSample of duplicate records:")
    print(df[df.duplicated()].head())
    rows_before = df.shape[0]
    df.drop_duplicates(inplace=True)
    print(f"\nRows before removal: {rows_before}")
    print(f"Rows after removal : {df.shape[0]}")
    print(f"\u2713 Removed {rows_before - df.shape[0]} duplicate records")
else:
    print("\u2713 No duplicate records found — dataset integrity confirmed")

Duplicate records found: 0
✓ No duplicate records found — dataset integrity confirmed


## 10. Reset Index

In [10]:
# After any row removal, the index may contain gaps (e.g. 0,1,3,5...).
# reset_index() renumbers rows consecutively starting from 0.
# drop=True prevents the old index being added as a new column.
# (Week 1 Practical 2 technique)

print(f"Last index before reset: {df.index[-1]}")
print(f"Expected last index    : {df.shape[0] - 1}")

df.reset_index(drop=True, inplace=True)

print(f"Last index after reset : {df.index[-1]}")
print("\u2713 Index reset successfully — consecutive row numbering confirmed")

Last index before reset: 11980
Expected last index    : 11980
Last index after reset : 11980
✓ Index reset successfully — consecutive row numbering confirmed


## 11. Find and Fix Whitespace Irregularities

In [11]:
# Whitespace irregularities (e.g. 'WEST  HAVEN') cause the same entity
# to be counted as separate categories in grouping and analysis.
# Regular expression \\s{2,} detects 2 or more consecutive spaces.
# Replaced with a single space; str.strip() removes leading/trailing spaces.
# (Week 1 Practical 2 technique: str.contains(), str.replace())

print("Checking 'Residence City' for irregular spacing:")
irregular = df['Residence City'].str.contains(r'\s{2,}', regex=True, na=False)
print(f"Records with irregular spacing: {irregular.sum()}")

if irregular.sum() > 0:
    print("\nSample of irregular records:")
    print(df[irregular]['Residence City'].head())
    df['Residence City'] = df['Residence City'].str.replace(r'\s{2,}', ' ', regex=True)
    df['Residence City'] = df['Residence City'].str.strip()
    print("\u2713 Irregular spacing corrected")
else:
    print("\u2713 No irregular spacing found")

Checking 'Residence City' for irregular spacing:
Records with irregular spacing: 1

Sample of irregular records:
1470    WEST  HAVEN
Name: Residence City, dtype: object
✓ Irregular spacing corrected


## 12. Data Transformation – Date Conversion and Temporal Feature Extraction

In [12]:
# The Date column is stored as a text string in the raw dataset.
# pd.to_datetime() converts it to Python datetime format.
# errors='coerce' converts any unparseable values to NaT (Not a Time).
# Four temporal features are extracted for downstream mining tasks:
#
# Year      : 2012-2023 — primary index for age drift analysis and regression
# Month     : 1-12     — seasonal pattern analysis
# DayOfWeek : 0-6       — 0=Monday, 6=Sunday; weekly pattern analysis
# Season    : Spring/Summer/Autumn/Winter — derived from Month mapping

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek

df['Season'] = df['Month'].map({
    12: 'Winter', 1: 'Winter',  2: 'Winter',
     3: 'Spring', 4: 'Spring',  5: 'Spring',
     6: 'Summer', 7: 'Summer',  8: 'Summer',
     9: 'Autumn', 10: 'Autumn', 11: 'Autumn'
})

print("\u2713 Temporal features created")
print(f"Date range : {int(df['Year'].min())} to {int(df['Year'].max())}")
print(f"\nSeason distribution:")
print(df['Season'].value_counts())

✓ Temporal features created
Date range : 2012 to 2023

Season distribution:
Season
Summer    3121
Autumn    3042
Spring    2967
Winter    2851
Name: count, dtype: int64


## 13. Data Transformation – Discretization (Age Groups)

In [13]:
# Convert continuous Age into 3 discrete categories using pd.cut().
# Bins: Young (0-30), Middle-aged (30-60), Senior (60+)
# The Senior category grew +1,375% between 2012 and 2023,
# making it the primary target class for classification (Objective 1).
# (Week 1 Practical 2 technique: pd.cut())

df['Age_Group'] = pd.cut(
    df['Age'],
    bins=[0, 30, 60, 100],
    labels=['Young (0-30)', 'Middle-aged (30-60)', 'Senior (60+)']
)

print("Age Group Distribution:")
print(df['Age_Group'].value_counts())
print(f"\nUnique groups: {df['Age_Group'].unique().tolist()}")
print("\u2713 Age discretization completed")

Age Group Distribution:
Age_Group
Middle-aged (30-60)    8664
Young (0-30)           2056
Senior (60+)           1261
Name: count, dtype: int64

Unique groups: ['Middle-aged (30-60)', 'Young (0-30)', 'Senior (60+)']
✓ Age discretization completed


## 14. Data Transformation – Z-Score Normalization for Age

In [14]:
# Z-score formula: z = (x - mean) / std
# z = 0    : value equals the population mean
# |z| > 3  : extreme outlier (more than 3 standard deviations from mean)
# Age_zscore flags statistically extreme ages for further analysis.
# (Week 1 Practical 2 technique: manual z-score formula)

df['Age_zscore'] = (df['Age'] - df['Age'].mean()) / df['Age'].std()

print("Age Z-Score Statistics:")
print(f"  Mean Age  : {df['Age'].mean():.2f} years")
print(f"  Std Dev   : {df['Age'].std():.2f}")
print(f"  Min z     : {df['Age_zscore'].min():.2f}")
print(f"  Max z     : {df['Age_zscore'].max():.2f}")

extreme = df[abs(df['Age_zscore']) > 3]
print(f"\nExtreme age outliers (|z| > 3): {len(extreme)} records")
if len(extreme) > 0:
    print(extreme[['Age', 'Age_zscore']])
    print("\nDecision: RETAINED — ages are biologically plausible (elderly overdose victims).")
    print("High z-scores retained as they represent genuine edge cases in the dataset.")


Age Z-Score Statistics:
  Mean Age  : 44.01 years
  Std Dev   : 12.68
  Min z     : -2.45
  Max z     : 3.39

Extreme age outliers (|z| > 3): 2 records
       Age  Age_zscore
3506  84.0    3.154236
8314  87.0    3.390870

Decision: RETAINED — ages are biologically plausible (elderly overdose victims).
High z-scores retained as they represent genuine edge cases in the dataset.


## 15. Data Transformation – MinMax Scaling for Age (Clustering Preparation)

In [15]:
# MinMax scaling rescales Age to range [0, 1].
# Formula: (x - min) / (max - min)
# Required for K-Means clustering:
# K-Means uses Euclidean distance — unscaled features with larger numerical
# ranges disproportionately dominate cluster assignments.
# Age_MinMax ensures Age contributes proportionally alongside binary
# drug features (already in [0, 1] range).
# Consistent with MinMax technique in Week 5 Practical 2.

scaler = MinMaxScaler()
df['Age_MinMax'] = scaler.fit_transform(df[['Age']])

print("MinMax Scaling Results:")
print(f"  Original Age range : {df['Age'].min():.0f} to {df['Age'].max():.0f}")
print(f"  Scaled range       : {df['Age_MinMax'].min():.2f} to {df['Age_MinMax'].max():.2f}")
print("✓ Age_MinMax created — ready for K-Means clustering")


MinMax Scaling Results:
  Original Age range : 13 to 87
  Scaled range       : 0.00 to 1.00
✓ Age_MinMax created — ready for K-Means clustering


## 16. Data Transformation – Label Encoding for Categorical Variables

In [16]:
# Machine learning algorithms (Decision Tree, Random Forest, K-Means)
# require numerical inputs — categorical text must be converted to integers.
# LabelEncoder assigns a unique integer to each unique category value.
# Original text columns are preserved alongside encoded columns.
# (Week 1 Practical 2 technique: sklearn LabelEncoder)

le = LabelEncoder()

# Sex encoding
print("Sex encoding:")
print(f"  Original unique values : {sorted(df['Sex'].unique().tolist())}")
df['Sex_Encoded'] = le.fit_transform(df['Sex'])
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"  Encoding mapping       : {mapping}")

# Race encoding
print("\nRace encoding:")
print(f"  Unique categories      : {df['Race'].nunique()}")
df['Race_Encoded'] = le.fit_transform(df['Race'])
print(f"  Encoded range          : 0 to {df['Race_Encoded'].max()}")

# Age_Group encoding
print("\nAge_Group encoding:")
print(f"  Original unique values : {df['Age_Group'].unique().tolist()}")
df['Age_Group_Encoded'] = le.fit_transform(df['Age_Group'])
ag_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"  Encoding mapping       : {ag_mapping}")

print("\n\u2713 All categorical variables encoded")

Sex encoding:
  Original unique values : ['Female', 'Male', 'Unknown', 'X']
  Encoding mapping       : {'Female': np.int64(0), 'Male': np.int64(1), 'Unknown': np.int64(2), 'X': np.int64(3)}

Race encoding:
  Unique categories      : 22
  Encoded range          : 0 to 21

Age_Group encoding:
  Original unique values : ['Middle-aged (30-60)', 'Young (0-30)', 'Senior (60+)']
  Encoding mapping       : {'Middle-aged (30-60)': np.int64(0), 'Senior (60+)': np.int64(1), 'Young (0-30)': np.int64(2)}

✓ All categorical variables encoded


## 17. Feature Engineering – Age Drift Indicators (Novel Research Contribution)

In [17]:
# This cell creates novel engineered features for this project's unique
# research angle: the systematic ageing of overdose victims (2012-2023).
#
# Key statistical finding: mean victim age rose from 40.77 (2012)
# to 47.99 (2023) — a +7.22 year drift in 11 years.
# Senior (60+) deaths grew from 20 (2012) to 295 (2023) = +1,375%.
#
# Features created:
# Mean_Age_That_Year              : Average victim age in that calendar year
# Age_Deviation_From_Yearly_Mean  : Victim age minus that year's mean age
# Is_Senior                       : 1 if Age >= 60, else 0
# Age_Era                         : Temporal era label based on Year

# Feature 1 & 2: Yearly mean age and per-record deviation
yearly_mean = df.groupby('Year')['Age'].mean().reset_index()
yearly_mean.columns = ['Year', 'Mean_Age_That_Year']
df = df.merge(yearly_mean, on='Year', how='left')
df['Age_Deviation_From_Yearly_Mean'] = df['Age'] - df['Mean_Age_That_Year']

# Feature 3: Senior binary flag
df['Is_Senior'] = (df['Age'] >= 60).astype(int)

# Feature 4: Age era
df['Age_Era'] = pd.cut(
    df['Year'],
    bins=[2011, 2015, 2019, 2023],
    labels=['Early Era (2012-2015)', 'Mid Era (2016-2019)', 'Late Era (2020-2023)']
)

print("✓ Age Drift Features Created")
print("\nYearly Mean Age (demonstrates the +7.22 year drift):")
print(df[['Year', 'Mean_Age_That_Year']].drop_duplicates().sort_values('Year').to_string(index=False))
print(f"\nAge Deviation range : {df['Age_Deviation_From_Yearly_Mean'].min():.2f} to {df['Age_Deviation_From_Yearly_Mean'].max():.2f}")
print(f"Senior cases (Age >= 60): {df['Is_Senior'].sum()} ({df['Is_Senior'].mean()*100:.1f}%)")
print(f"\nAge Era distribution:")
print(df['Age_Era'].value_counts().sort_index())


✓ Age Drift Features Created

Yearly Mean Age (demonstrates the +7.22 year drift):
 Year  Mean_Age_That_Year
 2012           40.771831
 2013           41.387755
 2014           41.582437
 2015           42.303155
 2016           42.082879
 2017           41.730250
 2018           42.761062
 2019           43.276667
 2020           43.708879
 2021           45.751312
 2022           46.632231
 2023           47.991711

Age Deviation range : -33.63 to 45.61
Senior cases (Age >= 60): 1509 (12.6%)

Age Era distribution:
Age_Era
Early Era (2012-2015)    2132
Mid Era (2016-2019)      4172
Late Era (2020-2023)     5677
Name: count, dtype: int64


## 18. Feature Engineering – Drug Count and Polydrug Indicator

In [18]:
# Convert drug columns to numeric integers before summing.
# 'Any Opioid' excluded — it is a meta-indicator (derived from other columns),
# including it would cause double-counting.
#
# Drug_Count : Total substances detected per case (0 to 9)
#              Used in clustering and regression feature sets
# Polydrug   : Binary flag — 1 if Drug_Count > 1, else 0
#              76.8% of cases involve multiple substances

binary_drug_cols = [
    'Heroin', 'Cocaine', 'Fentanyl', 'Fentanyl Analogue', 'Oxycodone',
    'Oxymorphone', 'Ethanol', 'Hydrocodone', 'Benzodiazepine', 'Methadone',
    'Meth/Amphetamine', 'Amphet', 'Tramad', 'Hydromorphone',
    'Morphine (Not Heroin)', 'Xylazine', 'Gabapentin', 'Opiate NOS'
]

# Convert to numeric integer (in case any string '0'/'1' values remain)
for col in binary_drug_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

df['Drug_Count'] = df[[c for c in binary_drug_cols if c in df.columns]].sum(axis=1)
df['Polydrug'] = (df['Drug_Count'] > 1).astype(int)

print("Drug Count Statistics:")
print(df['Drug_Count'].describe())
print(f"\nDrug Count distribution:")
print(df['Drug_Count'].value_counts().sort_index())
print(f"\nPolydrug cases : {df['Polydrug'].sum()} ({df['Polydrug'].mean()*100:.1f}%)")
print("\u2713 Drug count features created")

Drug Count Statistics:
count    11981.000000
mean         2.352475
std          1.112552
min          0.000000
25%          2.000000
50%          2.000000
75%          3.000000
max          9.000000
Name: Drug_Count, dtype: float64

Drug Count distribution:
Drug_Count
0     136
1    2644
2    4334
3    3170
4    1243
5     353
6      87
7       9
8       4
9       1
Name: count, dtype: int64

Polydrug cases : 9201 (76.8%)
✓ Drug count features created


## 19. Feature Engineering – Alone Death Indicator

In [19]:
# A death is classified as 'alone at home' when:
#   Injury Place == 'Residence'  AND  Location == 'Residence'
# This indicates the person was likely unsupervised at time of overdose.
# 37.0% of all cases (4,429 records) meet this criterion.
# Particularly relevant for the Senior age group, which has fewer
# social safety nets compared to younger demographics.

df['Died_Alone_At_Home'] = (
    (df['Injury Place'] == 'Residence') &
    (df['Location'] == 'Residence')
).astype(int)

print(f"Died Alone At Home:")
print(f"  Total cases : {df['Died_Alone_At_Home'].sum()} ({df['Died_Alone_At_Home'].mean()*100:.1f}%)")
print(f"\nBreakdown by Age Group:")
print(df.groupby('Age_Group', observed=True)['Died_Alone_At_Home'].sum())
print("✓ Alone death indicator created")


Died Alone At Home:
  Total cases : 4429 (37.0%)

Breakdown by Age Group:
Age_Group
Young (0-30)            773
Middle-aged (30-60)    3200
Senior (60+)            456
Name: Died_Alone_At_Home, dtype: int64
✓ Alone death indicator created


## 20. Data Reduction – Column Reduction

In [20]:
# Remove columns that are irrelevant or redundant for the four mining tasks.
# Justification per group:
#
# Geographic coordinates (3 cols):
#   ResidenceCityGeo, InjuryCityGeo, DeathCityGeo
#   -> Spatial coordinate strings; no spatial analysis planned
#
# Redundant temporal metadata (1 col):
#   Date Type -> Superseded by engineered Year/Month/DayOfWeek/Season features
#
# Unstructured free text (4 cols):
#   Description of Injury, Location if Other,
#   Other Significant Conditions, Cause of Death
#   -> Free text; not suitable for standard ML without NLP preprocessing
#
# Redundant drug indicators (5 cols):
#   Heroin death certificate (DC), Heroin/Morph/Codeine,
#   Other Opioid, Other, Manner of Death
#   -> Derivative or low-analytical-value; captured by the binary drug matrix

columns_to_drop = [
    'ResidenceCityGeo', 'InjuryCityGeo', 'DeathCityGeo',
    'Date Type', 'Description of Injury', 'Location if Other',
    'Other Significant Conditions ', 'Cause of Death',
    'Heroin death certificate (DC)', 'Heroin/Morph/Codeine',
    'Manner of Death', 'Other Opioid', 'Other'
]

cols_before = df.shape[1]
df.drop(columns=[c for c in columns_to_drop if c in df.columns], inplace=True)
cols_after = df.shape[1]

print(f"Columns before : {cols_before}")
print(f"Columns after  : {cols_after}")
print(f"Removed        : {cols_before - cols_after} redundant/irrelevant columns")
print("\u2713 Column reduction completed")

Columns before : 65
Columns after  : 52
Removed        : 13 redundant/irrelevant columns
✓ Column reduction completed


## 21. Data Reduction – Random Sampling (20%)

In [21]:
# Reduce dataset from 11,981 to ~2,396 rows using 20% random sampling.
# random_state=42 ensures full reproducibility:
#   -> Same sample produced every run
#   -> All three group members get identical data when they load the CSV
# Representativeness validated by comparing key distributions.
# (Week 1 Practical 2 technique: df.sample(frac=0.2, random_state=42))

print(f"Full dataset size : {df.shape[0]} rows")

df_sampled = df.sample(frac=0.2, random_state=42)
df_sampled.reset_index(drop=True, inplace=True)

print(f"Sampled size      : {df_sampled.shape[0]} rows")
print(f"Reduction         : {(1 - df_sampled.shape[0]/df.shape[0])*100:.1f}%")

print("\nRepresentativeness Validation:")
print(f"  Mean Age   — Full: {df['Age'].mean():.2f} | Sample: {df_sampled['Age'].mean():.2f}")
print(f"  % Senior   — Full: {df['Is_Senior'].mean()*100:.1f}% | Sample: {df_sampled['Is_Senior'].mean()*100:.1f}%")
print(f"  % Male     — Full: {(df['Sex']=='Male').mean()*100:.1f}% | Sample: {(df_sampled['Sex']=='Male').mean()*100:.1f}%")
print(f"  % Polydrug — Full: {df['Polydrug'].mean()*100:.1f}% | Sample: {df_sampled['Polydrug'].mean()*100:.1f}%")
print("\n\u2713 Sample is statistically representative of the full dataset")

Full dataset size : 11981 rows
Sampled size      : 2396 rows
Reduction         : 80.0%

Representativeness Validation:
  Mean Age   — Full: 44.01 | Sample: 44.01
  % Senior   — Full: 12.6% | Sample: 12.6%
  % Male     — Full: 74.3% | Sample: 73.1%
  % Polydrug — Full: 76.8% | Sample: 77.4%

✓ Sample is statistically representative of the full dataset


## 22. Save Cleaned Dataset

In [22]:
# Save the fully preprocessed dataset as CSV.
# This file is the cleaned output for all subsequent mining tasks.

output_filename = 'Accidental_Drug_Related_Deaths_CLEANED_V2.csv'
df_sampled.to_csv(output_filename, index=False)

print(f"✓ Cleaned dataset saved: {output_filename}")
print(f"  Final size : {df_sampled.shape[0]} rows x {df_sampled.shape[1]} columns")
print("\nPreview of Final Dataset (first 5 rows):")
print(df_sampled.head())
print("\nFinal Columns:")
for col in df_sampled.columns:
    print(f"  {col}")


✓ Cleaned dataset saved: Accidental_Drug_Related_Deaths_CLEANED_V2.csv
  Final size : 2396 rows x 52 columns

Preview of Final Dataset (first 5 rows):
        Date   Age     Sex     Race Ethnicity Residence City Residence County  \
0 2021-03-26  54.0    Male  Unknown       NaN     TORRINGTON       LITCHFIELD   
1 2018-08-18  51.0  Female    Black       NaN      WATERBURY        NEW HAVEN   
2 2016-10-15  33.0    Male    White       NaN      KNOXVILLE             KNOX   
3 2020-07-29  37.0    Male    White       NaN      NEW HAVEN        NEW HAVEN   
4 2018-11-13  37.0    Male    White       NaN      WATERBURY        NEW HAVEN   

  Residence State   Injury City Injury County  ... Sex_Encoded Race_Encoded  \
0              CT    TORRINGTON    LITCHFIELD  ...           1           19   
1              CT      HARTFORD      HARTFORD  ...           0            5   
2              TN  BEACON FALLS       Unknown  ...           1           20   
3              CT     NEW HAVEN     NEW HAVEN 

## 23. Data Preprocessing Summary

In [23]:
# Final confirmation of all preprocessing steps completed.
# Original : 11,981 rows x 48 columns
# Final    :  2,396 rows x 52 columns

print("=" * 80)
print("DATA PREPROCESSING PIPELINE — COMPLETED SUCCESSFULLY")
print("=" * 80)
print(f"\nOriginal dataset : 11,981 rows x 48 columns")
print(f"Final dataset    : {df_sampled.shape[0]} rows x {df_sampled.shape[1]} columns")

print("\nSteps completed:")
steps = [
    ("Cell 5",  "Age missing values filled via median imputation (2 values)"),
    ("Cell 6",  "Sex (9) and Race (57) missing values filled via mode imputation"),
    ("Cell 7",  "9 location columns filled with 'Unknown'"),
    ("Cell 8",  "19 drug columns binarised: blank->0, Y->1"),
    ("Cell 9",  "Duplicate records confirmed: 0 found"),
    ("Cell 10", "Index reset to consecutive numbering"),
    ("Cell 11", "1 whitespace irregularity corrected in Residence City"),
    ("Cell 12", "Date converted; Year, Month, DayOfWeek, Season extracted"),
    ("Cell 13", "Age discretised into Age_Group (Young/Middle-aged/Senior)"),
    ("Cell 14", "Age_zscore created; 2 outliers (|z|>3) retained"),
    ("Cell 15", "Age_MinMax created (range 0-1) for K-Means clustering"),
    ("Cell 16", "Sex_Encoded, Race_Encoded, Age_Group_Encoded created"),
    ("Cell 17", "Age drift features: Mean_Age_That_Year, Age_Deviation, Is_Senior, Age_Era"),
    ("Cell 18", "Drug_Count (0-9) and Polydrug binary indicator created"),
    ("Cell 19", "Died_Alone_At_Home binary indicator created"),
    ("Cell 20", "13 redundant/irrelevant columns removed"),
    ("Cell 21", "20% random sample: 11,981 -> 2,396 rows (representative, random_state=42)"),
    ("Cell 22", "Saved: Accidental_Drug_Related_Deaths_CLEANED_V2.csv"),
]
for cell_ref, step in steps:
    print(f"  ✓ [{cell_ref}] {step}")

print("\nKey engineered features (ageing overdose victim research angle):")
novel = [
    ("Mean_Age_That_Year",              "Average victim age per calendar year"),
    ("Age_Deviation_From_Yearly_Mean",  "Per-record deviation from yearly mean age"),
    ("Is_Senior",                       "Binary flag: Age >= 60"),
    ("Age_Era",                         "Temporal era grouping (Early / Mid / Late)"),
]
for feat, desc in novel:
    print(f"  ✓ {feat:<35} : {desc}")

print("\n" + "=" * 80)
print("ALL CELLS COMPLETED — NO ERRORS")
print("=" * 80)


DATA PREPROCESSING PIPELINE — COMPLETED SUCCESSFULLY

Original dataset : 11,981 rows x 48 columns
Final dataset    : 2396 rows x 52 columns

Steps completed:
  ✓ [Cell 5] Age missing values filled via median imputation (2 values)
  ✓ [Cell 6] Sex (9) and Race (57) missing values filled via mode imputation
  ✓ [Cell 7] 9 location columns filled with 'Unknown'
  ✓ [Cell 8] 19 drug columns binarised: blank->0, Y->1
  ✓ [Cell 9] Duplicate records confirmed: 0 found
  ✓ [Cell 10] Index reset to consecutive numbering
  ✓ [Cell 11] 1 whitespace irregularity corrected in Residence City
  ✓ [Cell 12] Date converted; Year, Month, DayOfWeek, Season extracted
  ✓ [Cell 13] Age discretised into Age_Group (Young/Middle-aged/Senior)
  ✓ [Cell 14] Age_zscore created; 2 outliers (|z|>3) retained
  ✓ [Cell 15] Age_MinMax created (range 0-1) for K-Means clustering
  ✓ [Cell 16] Sex_Encoded, Race_Encoded, Age_Group_Encoded created
  ✓ [Cell 17] Age drift features: Mean_Age_That_Year, Age_Deviation, Is_Sen